# Observability & Tracing

LangGOAP fires lifecycle hooks at every stage of the planning loop:
plan start/complete/failed, action start/complete, replans, goal
achievement, and A* search-tree expansions.  Three built-in tracers
turn these hooks into actionable observability:

| Tracer | Purpose |
|--------|----------|
| `LoggingTracer` | Routes events to stdlib `logging` — quick local debugging |
| `LangSmithTracer` | Emits a structured run tree to LangSmith — production monitoring |
| `MultiTracer` | Fans events to multiple tracers simultaneously |

This notebook demonstrates all three using the **Star News Finder**
pipeline — five LLM-powered actions that extract a person, look up
their star sign, retrieve a horoscope, find news, and compose a
writeup.  Real LLM calls make the traces interesting.

## Setup

Load environment variables and create the LLM + action set.

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY (and optionally LANGCHAIN_API_KEY)

# tutorial_examples lives under examples/tutorials/
sys.path.insert(0, str(Path.cwd().parent / "tutorials"))

from langchain_openai import ChatOpenAI

from langgoap import ActionSpec, GoalSpec, GoapGraph, successful_action_names
from langgoap.tracing import LoggingTracer, LangSmithTracer, MultiTracer
from tutorial_examples.agent_pattern_examples import star_news_actions

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
actions = star_news_actions(llm)
goal = GoalSpec(conditions={"writeup_complete": True})
world_state = {"has_user_input": True, "user_input": "Tell me about Taylor Swift, she's a Sagittarius"}

print(f"Actions: {[a.name for a in actions]}")
print(f"Goal:    {dict(goal.conditions)}")

Actions: ['extract_person', 'extract_star_sign', 'retrieve_horoscope', 'find_news_stories', 'star_news_writeup']
Goal:    {'writeup_complete': True}


## LoggingTracer — Local Debugging

`LoggingTracer` routes every hook to the `langgoap.tracing` logger at
`INFO` level.  We capture the output with a `StringIO` handler so it
appears inline in the notebook.

In [2]:
import io
import logging

# Capture log output into a string buffer
buf = io.StringIO()
handler = logging.StreamHandler(buf)
handler.setFormatter(logging.Formatter("%(levelname)s %(message)s"))

tracer_logger = logging.getLogger("langgoap.tracing")
tracer_logger.addHandler(handler)
tracer_logger.setLevel(logging.INFO)

tracer = LoggingTracer()
graph = GoapGraph(actions=actions, tracer=tracer)
result = graph.invoke(goal=goal, world_state=dict(world_state))

# Clean up handler
tracer_logger.removeHandler(handler)

print(f"Status: {result['status']}")
print(f"Actions: {successful_action_names(result)}")
print()
print("=== Traced Events ===")
print(buf.getvalue())

Status: goal_achieved
Actions: ['extract_person', 'extract_star_sign', 'retrieve_horoscope', 'find_news_stories', 'star_news_writeup']

=== Traced Events ===
INFO plan_start strategy=AStar goal=GoalSpec(conditions={'writeup_complete': True})
INFO plan_complete duration_ms=0.14 plan=Plan(actions=['extract_person', 'extract_star_sign', 'retrieve_horoscope', 'find_news_stories', 'star_news_writeup'], total_cost=6, steps=5)
INFO action_start name=extract_person
INFO action_complete result={'world_state': {'has_user_input': True, 'user_input': "Tell me about Taylor Swift, she's a Sagittarius", 'has_person': True, 'person': {'name': 'Taylor Swift', 'extracted_from': "Tell me about Taylor Swift, she's a Sagittarius"}}, 'current_step': 1, 'execution_history': [ActionResult(action_name='extract_person', success=True, state_before={'has_user_input': True, 'user_input': "Tell me about Taylor Swift, she's a Sagittarius"}, state_after={'has_user_input': True, 'user_input': "Tell me about Taylor Swi

The log shows the full lifecycle:

1. `plan_start` — A* begins with strategy and goal
2. `plan_complete` — plan found with duration
3. `action_start` / `action_complete` — each action executes
4. `goal_achieved` — final state satisfies the goal

This is enough for local debugging.  For production, read on.

## LangSmithTracer — Production Monitoring

`LangSmithTracer` emits each planning cycle as a LangSmith run tree:

- **Root run** (`goap_plan`, kind `chain`) — one per planning cycle
- **Child runs** (`goap_action:<name>`, kind `tool`) — one per executed action
- **Tags** — `goap_strategy:astar`, `goap_outcome:goal_achieved`

This tracer is complementary to LangGraph's automatic node-level tracing.
LangGraph traces the graph structure; `LangSmithTracer` traces GOAP domain
events (plans, replans, goal achievement) that node spans don't see.

To see traces in the [LangSmith UI](https://smith.langchain.com):

```bash
export LANGCHAIN_TRACING_V2=true
export LANGCHAIN_API_KEY=<your-key>
export LANGCHAIN_PROJECT=langgoap-tracing-demo
```

In [3]:
langsmith_available = bool(
    os.environ.get("LANGCHAIN_API_KEY") or os.environ.get("LANGSMITH_API_KEY")
)

if langsmith_available:
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_PROJECT", "langgoap-tracing-demo")

    ls_tracer = LangSmithTracer(project_name="langgoap-tracing-demo")
    graph = GoapGraph(actions=actions, tracer=ls_tracer)
    result = graph.invoke(goal=goal, world_state=dict(world_state))

    print(f"Status: {result['status']}")
    print(f"Actions: {successful_action_names(result)}")
    print()
    print("Trace sent to LangSmith project 'langgoap-tracing-demo'.")
    print("Open https://smith.langchain.com to view the run tree.")
else:
    print("LANGCHAIN_API_KEY not set — skipping LangSmith demo.")
    print("Set LANGCHAIN_API_KEY in your .env to enable LangSmith tracing.")

LANGCHAIN_API_KEY not set — skipping LangSmith demo.
Set LANGCHAIN_API_KEY in your .env to enable LangSmith tracing.


### Run tree structure

In the LangSmith UI the trace appears as:

```
goap_plan (chain) [goap_strategy:astar, goap_outcome:goal_achieved]
  ├── goap_action:extract_person (tool)
  ├── goap_action:extract_star_sign (tool)
  ├── goap_action:retrieve_horoscope (tool)
  ├── goap_action:find_news_stories (tool)
  └── goap_action:star_news_writeup (tool)
```

Use the `goap_outcome:*` tag to filter runs in the UI:
- `goap_outcome:goal_achieved` — successful completions
- `goap_outcome:error` — planning failures
- `goap_outcome:superseded` — runs that triggered a replan

## MultiTracer — Dual Output

`MultiTracer` composes multiple tracers so events fan out to all of
them.  Each inner tracer is exception-safe — one broken tracer cannot
stop the others from receiving events.

In [4]:
buf2 = io.StringIO()
handler2 = logging.StreamHandler(buf2)
handler2.setFormatter(logging.Formatter("%(levelname)s %(message)s"))
tracer_logger.addHandler(handler2)
tracer_logger.setLevel(logging.INFO)

log_tracer = LoggingTracer()

if langsmith_available:
    ls_tracer2 = LangSmithTracer(project_name="langgoap-tracing-demo")
    multi = MultiTracer([log_tracer, ls_tracer2])
    print("MultiTracer: LoggingTracer + LangSmithTracer")
else:
    multi = MultiTracer([log_tracer])
    print("MultiTracer: LoggingTracer only (no LANGCHAIN_API_KEY)")

graph = GoapGraph(actions=actions, tracer=multi)
result = graph.invoke(goal=goal, world_state=dict(world_state))

tracer_logger.removeHandler(handler2)

print(f"\nStatus: {result['status']}")
print(f"Actions: {successful_action_names(result)}")
print()
print("=== LoggingTracer output (via MultiTracer) ===")
print(buf2.getvalue())

MultiTracer: LoggingTracer only (no LANGCHAIN_API_KEY)



Status: goal_achieved
Actions: ['extract_person', 'extract_star_sign', 'retrieve_horoscope', 'find_news_stories', 'star_news_writeup']

=== LoggingTracer output (via MultiTracer) ===
INFO plan_start strategy=AStar goal=GoalSpec(conditions={'writeup_complete': True})
INFO plan_complete duration_ms=0.18 plan=Plan(actions=['extract_person', 'extract_star_sign', 'retrieve_horoscope', 'find_news_stories', 'star_news_writeup'], total_cost=6, steps=5)
INFO action_start name=extract_person
INFO action_complete result={'world_state': {'has_user_input': True, 'user_input': "Tell me about Taylor Swift, she's a Sagittarius", 'has_person': True, 'person': {'name': 'Taylor Swift', 'extracted_from': "Tell me about Taylor Swift, she's a Sagittarius"}}, 'current_step': 1, 'execution_history': [ActionResult(action_name='extract_person', success=True, state_before={'has_user_input': True, 'user_input': "Tell me about Taylor Swift, she's a Sagittarius"}, state_after={'has_user_input': True, 'user_input':

## Writing a Custom Tracer

Any object that implements the `PlanningTracer` protocol works as a tracer.
The protocol is a set of `on_*` and `aon_*` hooks — implement only the ones
you need (the rest default to no-ops).  Here's a minimal counter tracer:

In [5]:
from langgoap.tracing import NullTracer


class CounterTracer(NullTracer):
    """Counts planning events — extend NullTracer to get default no-ops."""

    def __init__(self) -> None:
        self.plans = 0
        self.actions_run = 0
        self.goals_achieved = 0

    def on_plan_complete(self, plan, duration_ms):
        self.plans += 1

    def on_action_complete(self, result):
        self.actions_run += 1

    def on_goal_achieved(self, final_state):
        self.goals_achieved += 1


counter = CounterTracer()
graph = GoapGraph(actions=actions, tracer=counter)
result = graph.invoke(goal=goal, world_state=dict(world_state))

print(f"Plans found:     {counter.plans}")
print(f"Actions run:     {counter.actions_run}")
print(f"Goals achieved:  {counter.goals_achieved}")

Plans found:     1
Actions run:     5
Goals achieved:  1


## Summary

- **`LoggingTracer`** — zero-config local debugging via stdlib logging.
- **`LangSmithTracer`** — production-grade run trees with outcome tags
  for filtering.  Complementary to LangGraph's built-in node tracing.
- **`MultiTracer`** — compose tracers for dual output (e.g. logging +
  LangSmith).  Exception-safe by design.
- **Custom tracers** — extend `NullTracer` and override only the hooks
  you need.  The `PlanningTracer` protocol has full sync/async parity.

Every scenario here is verified by
[`tests/integration/test_tracing_end_to_end.py`](../../tests/integration/test_tracing_end_to_end.py).